In [8]:
import os

# Convert notes to Roman numerals
c2roman = {
	'Cb':	'bI',
	'C':	'I',
	'C#':	'#I',
	'Db':	'bII',
	'D':	'II',
	'D#':	'#II',
	'Eb':	'bIII',
	'E':	'III',
	'F':	'IV',
	'F#':	'#IV',
	'Gb':	'bV',
	'G':	'V',
	'G#':	'#V',
	'Ab':	'bVI',
	'A':	'VI',
	'A#':	'#VI',
	'Bb':	'bVII',
	'B':	'VII',
	'B#':	'#VII'
}

# Convert raw chord to tuple of (root, quality, inversion)
def parse_chord(raw_chord, ignore_inversion=False):
	if raw_chord[0] == '.':
		return ('.', None, None)
	chord, inversion = raw_chord.split('/')
	if len(chord) > 1 and chord[1] in ['b', '#']:
		root = chord[:2]
		quality = chord[2:]
	else:
		root = chord[0]
		quality = chord[1:]
	if quality == '':
		quality = 'maj'
	inversion = None if ignore_inversion else int(inversion)

	return (root, quality, inversion)

# Diaplsy chord in Roman numal notation
def display_chord(chord):
	root, quality, inversion = chord
	if root == '.':
		return '.'
	if inversion:
		return(c2roman[root] + quality + '/' + str(inversion))
	else:
		return(c2roman[root] + quality)

# Read chords form file
def read_chords(filename, ignore_inversion=False):
	if not os.path.exists(filename):
		print(f"File {filename} does not exist.")
		return []


	with open(filename, 'r') as file:
		if ignore_inversion:
			chords = [parse_chord(c, ignore_inversion=True) for c in file.read().splitlines()]
		else:
			chords = [parse_chord(c, ignore_inversion) for c in file.read().splitlines()]
	return chords

In [70]:
bach_chords = read_chords(os.path.join('bach', 'wtc1-prelude1.txt'))
beethoven2_chords = read_chords(os.path.join('beethoven', 'sonata08-mv02.txt'))
beethoven3_chords = read_chords(os.path.join('beethoven', 'sonata08-mv03.txt'))
chopin_chords = read_chords(os.path.join('chopin', 'etude10-01.txt'))

chords = bach_chords + beethoven2_chords + beethoven3_chords + chopin_chords
majorkey_chords = bach_chords + beethoven2_chords + chopin_chords

In [9]:
from collections import defaultdict
import random

class UnigramChordModel:
	def __init__(self, chords):
		self.chords = defaultdict(list)
		for chord in chords:
			if chord[0] != '.':
				self.chords[chord[0]].append(chord)
	def predict_next(self, chord):
		next_chords = self.chords.get(chord[0], [])
		if not next_chords:
			return None
		return random.choice(next_chords)

class BigramChordModel:
	def __init__(self, chords):
		self.bigrams = defaultdict(list)
		prev = None
		for chord in chords:
			if prev is not None:
				self.bigrams[prev].append(chord)
			prev = chord

	def predict_next(self, chord):
		next_chords = self.bigrams.get(chord, [])
		if not next_chords:
			return None
		return random.choice(next_chords)

class TrigramChordModel:
	def __init__(self, chords):
		self.trigrams = defaultdict(list)
		prev1, prev2 = None, None
		for chord in chords:
			if prev1 is not None and prev2 is not None:
				self.trigrams[(prev1, prev2)].append(chord)
			prev1, prev2 = prev2, chord

	def predict_next(self, prev1, prev2):
		next_chords = self.trigrams.get((prev1, prev2), [])
		if not next_chords:
			return None
		return random.choice(next_chords)

class BackoffChordModel:
	def __init__(self, chords):
		self.unigram = UnigramChordModel(chords)
		self.bigram = BigramChordModel(chords)
		self.trigram = TrigramChordModel(chords)

	def predict_next(self, prev1, prev2):
		# Try trigram
		next_chord = self.trigram.predict_next(prev1, prev2)
		if next_chord:
			return next_chord
		# Try bigram
		next_chord = self.bigram.predict_next(prev2)
		if next_chord:
			return next_chord
		# Try unigram
		return self.unigram.predict_next(prev2)

In [56]:
# Test bigram
bigram_model = BigramChordModel(majorkey_chords)
current_chord = parse_chord('Em/0')
predicted_next = bigram_model.predict_next(current_chord)
print("Given chord:", display_chord(current_chord))
if predicted_next:
	print("Predicted next chord:", display_chord(predicted_next))
else:
	print("No prediction available.")

Given chord: IIIm
Predicted next chord: VIm7/1


In [76]:
# Test trigram
trigram_model = TrigramChordModel(chords)
current_chord1 = parse_chord('F/0')
current_chord2 = parse_chord('F#m7b5/0')
predicted_next = trigram_model.predict_next(current_chord1, current_chord2)
print("Given chords:", display_chord(current_chord1), display_chord(current_chord2))
if predicted_next:
	print("Predicted next chord:", display_chord(predicted_next))
else:
	print("No prediction available.")

Given chords: IVmaj #IVm7b5
Predicted next chord: Vmaj


In [77]:
# Example usage:
backoff_model = BackoffChordModel(majorkey_chords)
# For prediction, provide two previous chords (e.g., chords[-2], chords[-1])
predicted = backoff_model.predict_next(chords[-2], chords[-1])
print("Predicted next chord:", display_chord(predicted) if predicted else "No prediction available.")

Predicted next chord: Imaj


In [83]:
def gererate_chord_progression(model, start_chord, length=8):
	progression = [(display_chord(start_chord))]
	prev1, prev2 = None, start_chord
	for _ in range(length - 1):
		next_chord = model.predict_next(prev1, prev2)
		if next_chord is None:
			break
		progression.append(display_chord(next_chord))
		prev1, prev2 = prev2, next_chord
	return progression

def print_progression(progression):
	print(' '.join(display(chord) for chord in progression))


In [90]:
I = parse_chord('C/0')
for _ in range(4):
	print(*gererate_chord_progression(model=backoff_model, start_chord=I, length=8), sep='\t')

Imaj	V7/3	Imaj/1	V7/1	Imaj	Vmaj/1	VIm	II7/2
Imaj	.	V7	Imaj	.	V7	Imaj	.
Imaj	.	Imaj	IVmaj/1	#IVm7b5/1	Vsus4	Vmaj	Imaj
Imaj	IVmaj	#IVm7b5	Vmaj	II7	V7sus4	V7	Imaj


In [89]:
V = parse_chord('G/0')
for _ in range(4):
	print(*gererate_chord_progression(model=backoff_model, start_chord=V, length=8), sep='\t')

Vmaj	V7	Imaj	.	V7	Imaj	.	Im
Vmaj	.	#Idim7/2	IIm/1	VIIdim7/2	Imaj/1	.	IVmaj7/3
Vmaj	.	II7	V7sus4	V7	I7	IVmaj/2	IIm7
Vmaj	.	VIIm7b5/2	V7/3	Imaj/1	VI7	IIm	V7
